In [1]:
# Necessary imports
import re
import emoji
from deep_translator import GoogleTranslator
import random
import pandas as pd
import numpy as np
from multiprocessing import  Pool
import time

In [2]:
# Function to remove emojis in text, since these conflict during translation
def remove_emoji(text):
    return emoji.get_emoji_regexp().sub(u'', text)


def approximate_emoji_insert(string, index,char):
    if(index<(len(string)-1)):
        
        while(string[index]!=' ' ):
            if(index+1==len(string)):
                break
            index=index+1
        return string[:index] + ' '+char + ' ' + string[index:]
    else:
        return string + ' '+char + ' ' 
    


def extract_emojis(str1):
    try:
        return [(c,i) for i,c in enumerate(str1) if c in emoji.UNICODE_EMOJI]
    except AttributeError:
        return []

In [3]:
import time
import pandas as pd
from deep_translator import GoogleTranslator

def translate(x):
    emoji_list = extract_emojis(x)
    
    # We will let the error crash the notebook if Google blocks you!
    translated_text = GoogleTranslator(source='auto', target='en').translate(x)
    
    for ele in emoji_list:
        translated_text = approximate_emoji_insert(translated_text, ele[1], ele[0])
    return translated_text

def add_features(df):
    translated_text = []
    for index, row in df.iterrows():
        translated_text.append(translate(row['text']))
        # Add a half-second delay so Google doesn't block your IP!
        time.sleep(0.5)
        
    df = df.copy()
    df["translated"] = translated_text
    return df

def parallelize_dataframe(df, func, n_cores=4):
    return func(df)


In [4]:
import glob 
train_files = glob.glob('train/*.csv')
test_files = glob.glob('test/*.csv')
val_files = glob.glob('val/*.csv')
files= train_files+test_files+val_files

In [5]:
from tqdm.notebook import tqdm

size=10

for file in files:
    print(f"Processing {file}...")
    wp_data=pd.read_csv(file)
    
    # If it's an English dataset, skip the translation API to save massive amounts of time!
    if 'English' in file:
        wp_data['translated'] = wp_data['text']
        file_name='translated_'+file.split('/')[-1]
        wp_data.to_csv(file_name, index=False)
        print("Skipped translation (Already English).")
        continue
        
    list_df=[]
    for i in tqdm(range(0, len(wp_data), size)):
        df_new=parallelize_dataframe(wp_data[i:i+size],add_features,n_cores=4)
        list_df.append(df_new)
        
    df_translated=pd.concat(list_df,ignore_index=True)
    file_name='translated_'+file.split('/')[-1]
    df_translated.to_csv(file_name, index=False)


Processing train/Portugese_7a_full.csv...


  0%|          | 0/397 [00:00<?, ?it/s]

TooManyRequests: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function